# Demo: FP-Growth* — Khai thác Tập Phổ Biến




## 1. Nạp module

In [1]:
import Pkg
Pkg.activate(joinpath(@__DIR__, ".."))   # kích hoạt môi trường Project.toml
include(joinpath(@__DIR__, "..", "src", "FPGrowthStar.jl"))
using .FPGrowthStar
using Random
Random.seed!(42)   # cố định seed để kết quả tái lập được

  Activating project at `c:\Users\ADMIN\Desktop\itemset`


TaskLocalRNG()

## 2. Đọc dữ liệu & khai phá tập phổ biến

`example1.txt` là CSDL đồ chơi . Mỗi dòng là một giao dịch gồm các item (số nguyên) cách nhau bởi khoảng trắng — đúng định dạng SPMF.

In [2]:
transactions = read_spmf(joinpath(@__DIR__, "..", "data", "toy", "example1.txt"))
println("Số giao dịch: ", length(transactions))
transactions

Số giao dịch: 7


7-element Vector{Vector{Int64}}:
 [1, 2, 3]
 [1, 2, 4]
 [1, 3, 4]
 [2, 3, 4]
 [1, 2, 3, 4]
 [1, 3]
 [2, 4]

In [3]:
minsup = 3
results = fpgrowth_star(transactions, minsup)
println("Số tập phổ biến (minsup=$minsup): ", length(results))
write_results(stdout, results)

Số tập phổ biến (minsup=3): 10
1 #SUP: 5
2 #SUP: 5
3 #SUP: 5
4 #SUP: 5
1 2 #SUP: 3
1 3 #SUP: 4
2 3 #SUP: 3
1 4 #SUP: 3
2 4 #SUP: 4
3 4 #SUP: 3


## 3. Trực quan hóa FP-tree

Cấu trúc dữ liệu cốt lõi của thuật toán. `build_fptree` dựng cây nén từ toàn bộ CSDL.

In [4]:
tree = build_fptree(transactions, minsup)
println("Thứ tự item (giảm dần theo support): ", tree.ordered_items)
println()
print_tree(tree.root)

Thứ tự item (giảm dần theo support): [1, 2, 3, 4]

root
  item=1 count=5
    item=2 count=3
      item=3 count=2
        item=4 count=1
      item=4 count=1
    item=3 count=2
      item=4 count=1
  item=2 count=2
    item=3 count=1
      item=4 count=1
    item=4 count=1


## 4. Trường hợp đặc biệt: FP-tree nhánh đơn (single-path)

Ví dụ 2 (`example2.txt`) là tình huống đặc biệt **single-path**: FP-tree chỉ có một nhánh. Khi đó FP-Growth\* không đệ quy nữa mà **liệt kê trực tiếp** mọi tập con khác rỗng của nhánh — đúng $2^n-1$ tập phổ biến (với $n$ là số item trên nhánh).

In [5]:
t2 = read_spmf(joinpath(@__DIR__, "..", "data", "toy", "example2.txt"))
r_star = fpgrowth_star(t2, 1)
n_items = length(unique(vcat(t2...)))
println("Số item phân biệt: ", n_items)
println("FP-Growth* tìm được: ", length(r_star), " tập phổ biến")
println("Kỳ vọng 2^n - 1   : ", 2^n_items - 1)
println("Khớp? ", length(r_star) == 2^n_items - 1)

Số item phân biệt: 5
FP-Growth* tìm được: 31 tập phổ biến
Kỳ vọng 2^n - 1   : 31
Khớp? true


## 5. Ứng dụng: sinh luật kết hợp (Market Basket Analysis)

Từ tập phổ biến, hàm `association_rules` của nhóm (`src/rules.jl`) sinh luật $X \Rightarrow Y$ với **support**, **confidence** và **lift**.

Dưới đây minh họa trên CSDL đồ chơi. Ứng dụng đầy đủ trên CSDL bán lẻ thực (Retail, 88K giao dịch → ~6000 luật) xem `test/test_rules.jl`, kết quả xuất ra `results/rules_retail.csv`.

In [6]:
rules = association_rules(results, length(transactions); minconf=0.6)
println("Số luật (minconf=0.6): ", length(rules))
println("\nTop luật kết hợp (theo lift):")
for r in first(rules, min(10, length(rules)))
    println("  ", r.antecedent, " => ", r.consequent,
            "  (sup=", r.support,
            ", conf=", round(r.confidence, digits=2),
            ", lift=", round(r.lift, digits=2), ")")
end

Số luật (minconf=0.6): 12

Top luật kết hợp (theo lift):
  [4] => [2]  (sup=4, conf=0.8, lift=1.12)
  [2] => [4]  (sup=4, conf=0.8, lift=1.12)
  [3] => [1]  (sup=4, conf=0.8, lift=1.12)
  [1] => [3]  (sup=4, conf=0.8, lift=1.12)
  [4] => [3]  (sup=3, conf=0.6, lift=0.84)
  [3] => [4]  (sup=3, conf=0.6, lift=0.84)
  [4] => [1]  (sup=3, conf=0.6, lift=0.84)
  [1] => [4]  (sup=3, conf=0.6, lift=0.84)
  [3] => [2]  (sup=3, conf=0.6, lift=0.84)
  [2] => [3]  (sup=3, conf=0.6, lift=0.84)
